# Pure-Space Vecchia QC vs No-QC, July 3 2024, 8 Hours

This notebook compares the same pure-space Bessel Matérn cluster Vecchia model with and without extreme whitened-residual QC.

Design:
- data: July 3, 2024, first 8 hourly slots in the `[-3, 2] x [121, 131]` domain;
- model: pure-space anisotropic Bessel Matérn, `2x4` outer tiles;
- Vecchia: `4x4` target clusters, conditioning on 2 previous max-min cluster blocks;
- mean: `constant + centered latitude` by default;
- arms:
  - `no_qc`: fit once with `--outlier-whitened-threshold 0`;
  - `qc_w10`: fit once, compute detrended Vecchia whitened residuals, set `|w| > 10` to missing, refit.

The whitened residual is computed after GLS mean detrending: `w = L_i^{-1}(y_i - X_i beta_hat)` on each Vecchia local conditional block, then target rows are mapped back to observations.

## Screenshot Sanity Note

The screenshot you attached is not mainly a CSV-format problem; it is mostly terminal line wrapping. The more important signal is this pattern:

```text
success=False
message=STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT
outlier_whitened_threshold=10
n_qc_removed=0
qc_max_abs_whitened=blank
```

Before the helper patch, QC was skipped whenever the first optimizer result had `success=False`, even if it had finite `raw_params` and finite loss. That means max-iteration-limit fits could show threshold 10 but no whitening QC was actually computed. The scripts have now been patched so finite initial fits can still be used for QC diagnostics/refit.

In [ ]:
import os
import sys
import time
import contextlib
import io
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

PROJECT = Path('/Users/joonwonlee/Documents/GEMS_TCO-1')
SRC_DIR = PROJECT / 'src'
SCRIPT_DIR = PROJECT / 'Exercises/st_model/day/amarel_simulation/pure_space'
REFERENCE_NOTEBOOK = PROJECT / 'Exercises/st_model/day/local_computer/space_time/vecchia_cluster_batch/gpu_vecc_real_cluster_corridor_052426.ipynb'

sys.path.insert(0, str(SRC_DIR))
sys.path.insert(0, str(SCRIPT_DIR))

from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data_dynamic_processed

import fit_july2024_bessel_smooth_full_likelihood_tiles_2x4 as full_mod
import fit_july2024_bessel_smooth_vecchia_cluster_4x4_cond2_tiles_2x4 as vec_mod

print('reference:', REFERENCE_NOTEBOOK)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('script:', vec_mod.__file__)

In [ ]:
# Experiment controls.
YEAR = '2024'
MONTH = 7
DAY = '2024-07-03'
N_HOURS = 8

LAT_RANGE = [-3, 2]
LON_RANGE = [121, 131]
LAT_LON_RESOLUTION = [1, 1]

NUGGET_MODE = 'free'       # change to 'fixed0' if you want the no-nugget arm
MEAN_DESIGN = 'lat'
TILE_Y = 2
TILE_X = 4
TILE_MAX_POINTS = 0        # 0 = use all points per tile for Vecchia
MIN_TILE_POINTS = 200
TILE_WORKERS = 4           # set to 1 if multiprocessing in Jupyter is annoying

CLUSTER_BLOCK_SHAPE = (4, 4)
CLUSTER_NEIGHBOR_BLOCKS = 2
TARGET_CHUNK_SIZE = 128
MIN_TARGET_POINTS = 1

RANGE_LAT_INIT = 0.35
RANGE_LON_INIT = 0.35
SMOOTH_INIT = 0.5
SMOOTH_MIN = 0.05
SMOOTH_MAX = 2.5
RANGE_MIN = 0.03
RANGE_MAX = 5.0
JITTER = 1e-6
N_RESTARTS = 1
MAXITER = 80               # raised from 20 after repeated iteration-limit stops
MAXFUN = 0
MAXLS = 20
MAXCOR = 20
OPTIMIZER_METHOD = 'L-BFGS-B'

QC_THRESHOLD = 10.0
ARMS = [
    ('no_qc', 0.0),
    ('qc_w10', QC_THRESHOLD),
]

RUN_FITS = True
FORCE_REFIT = False
SUPPRESS_PRECOMPUTE_PRINTS = True

OUT_DIR = PROJECT / 'Exercises/st_model/day/local_computer/pure_space/diagnostics/qc_vs_noqc_july03_8h'
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = OUT_DIR / f'july03_8h_vecchia_cond2_{NUGGET_MODE}_qc_vs_noqc_tile_fits.csv'
PAIRED_CSV = OUT_DIR / f'july03_8h_vecchia_cond2_{NUGGET_MODE}_qc_vs_noqc_paired_delta.csv'
FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('outputs:', OUT_DIR)

## Load July Data

This follows the reference notebook's real-data loading pattern: `load_maxmin_ordered_data_bymonthyear(..., is_whittle=True)` avoids building the old point-level max-min neighbor map, because the cluster Vecchia model builds cluster order internally.

In [ ]:
loader = load_data_dynamic_processed(config.mac_data_load_path)

df_map, _, _, monthly_mean = loader.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=LAT_LON_RESOLUTION,
    mm_cond_number=1,
    years_=[YEAR],
    months_=[MONTH],
    lat_range=LAT_RANGE,
    lon_range=LON_RANGE,
    is_whittle=True,
)

key_idx = sorted(df_map)
print('n hourly slots:', len(key_idx))
print('monthly_mean:', monthly_mean)
print('first key:', key_idx[0], 'last key:', key_idx[-1])
print('columns:', list(df_map[key_idx[0]].columns))

In [ ]:
def parsed_timestamp(key):
    try:
        return full_mod.parse_gems_hour_key(str(key))
    except Exception:
        return None

want_day = pd.Timestamp(DAY).date()
selected_keys = [
    k for k in key_idx
    if parsed_timestamp(k) is not None and parsed_timestamp(k).date() == want_day
]

if len(selected_keys) < N_HOURS:
    # Reference notebook convention: 8 slots per day, zero-based day index.
    day_offset = (pd.Timestamp(DAY).day - 1) * 8
    selected_keys = key_idx[day_offset:day_offset + N_HOURS]
else:
    selected_keys = selected_keys[:N_HOURS]

if len(selected_keys) != N_HOURS:
    raise RuntimeError(f'Expected {N_HOURS} selected keys, got {len(selected_keys)}')

selected_info = pd.DataFrame({
    'hour_slot': np.arange(len(selected_keys)),
    'hour_key': selected_keys,
    'timestamp': [parsed_timestamp(k) for k in selected_keys],
    'n_rows': [len(df_map[k]) for k in selected_keys],
    'n_valid_o3': [int(pd.to_numeric(df_map[k]['ColumnAmountO3'], errors='coerce').notna().sum()) for k in selected_keys],
})
selected_info

In [ ]:
def assert_grid_order_consistent(keys):
    base_coords = df_map[keys[0]][['Latitude', 'Longitude']].to_numpy(dtype=np.float64)
    for k in keys[1:]:
        coords = df_map[k][['Latitude', 'Longitude']].to_numpy(dtype=np.float64)
        if coords.shape != base_coords.shape or not np.allclose(coords, base_coords, equal_nan=True):
            raise RuntimeError(f'Grid coordinate order differs at {k}; cluster local-index mapping may not be reusable.')
    return base_coords

base_grid_coords_np = assert_grid_order_consistent(selected_keys)
print('grid order consistency passed')
print('base grid shape:', base_grid_coords_np.shape)
print('unique lat/lon:', len(np.unique(base_grid_coords_np[:,0])), len(np.unique(base_grid_coords_np[:,1])))

## Fit Helpers

The fit uses the same helper functions as the Amarel pure-space Vecchia script, with only `outlier_whitened_threshold` changed between arms. The QC arm computes the first-fit Vecchia whitened residuals after GLS mean detrending, marks `|w| > 10` as `NaN`, rebuilds the cluster batches, and refits.

In [ ]:
def make_args(qc_threshold):
    return SimpleNamespace(
        time_col='hour',
        x_col='Longitude',
        y_col='Latitude',
        value_col='ColumnAmountO3',
        qa_col='',
        qa_min=None,
        coords='raw',
        lat_range=list(LAT_RANGE),
        lon_range=list(LON_RANGE),
        tile_y=int(TILE_Y),
        tile_x=int(TILE_X),
        min_tile_points=int(MIN_TILE_POINTS),
        tile_max_points=int(TILE_MAX_POINTS),
        tile_workers=int(TILE_WORKERS),
        sample_seed=202407,
        cluster_block_shape=tuple(CLUSTER_BLOCK_SHAPE),
        cluster_neighbor_blocks=int(CLUSTER_NEIGHBOR_BLOCKS),
        target_chunk_size=int(TARGET_CHUNK_SIZE),
        min_target_points=int(MIN_TARGET_POINTS),
        nugget_mode=str(NUGGET_MODE),
        mean_design=str(MEAN_DESIGN),
        range_lat_init=float(RANGE_LAT_INIT),
        range_lon_init=float(RANGE_LON_INIT),
        smooth_init=float(SMOOTH_INIT),
        nugget_init=None,
        smooth_min=float(SMOOTH_MIN),
        smooth_max=float(SMOOTH_MAX),
        range_min=float(RANGE_MIN),
        range_max=float(RANGE_MAX),
        jitter=float(JITTER),
        n_restarts=int(N_RESTARTS),
        maxiter=int(MAXITER),
        maxfun=int(MAXFUN),
        maxls=int(MAXLS),
        maxcor=int(MAXCOR),
        optimizer_method=str(OPTIMIZER_METHOD),
        outlier_whitened_threshold=float(qc_threshold),
    )


def fit_one_hour_arm(hour_slot, hour_key, condition, qc_threshold):
    args = make_args(qc_threshold)
    df_hour = df_map[hour_key].copy()
    # df_map is already one hourly frame and has no explicit time column;
    # the Amarel helper expects one because script-read hourly tables add it.
    df_hour['hour'] = str(hour_key)
    coords, values, _, meta = full_mod.prepare_hour_data(df_hour, args)
    tile_id, tile_meta = full_mod.assign_tiles(coords, int(args.tile_y), int(args.tile_x))

    t0 = time.time()
    if SUPPRESS_PRECOMPUTE_PRINTS:
        with contextlib.redirect_stdout(io.StringIO()):
            tile_df = vec_mod.fit_tiles_for_hour(coords, values, tile_id, tile_meta, args)
    else:
        tile_df = vec_mod.fit_tiles_for_hour(coords, values, tile_id, tile_meta, args)
    elapsed = time.time() - t0

    ts = parsed_timestamp(hour_key)
    tile_df.insert(0, 'condition', condition)
    tile_df.insert(1, 'qc_threshold', float(qc_threshold))
    tile_df.insert(2, 'day', DAY)
    tile_df.insert(3, 'hour_slot_selected', int(hour_slot))
    tile_df.insert(4, 'hour_key', str(hour_key))
    tile_df.insert(5, 'timestamp', '' if ts is None else ts.strftime('%Y-%m-%dT%H:%M:%SZ'))
    tile_df.insert(6, 'elapsed_s_hour_arm', elapsed)
    tile_df.insert(7, 'n_hour_points', int(len(values)))
    for k, v in meta.items():
        tile_df[f'data_{k}'] = v
    return tile_df


def run_experiment():
    rows = []
    for hour_slot, hour_key in enumerate(selected_keys):
        for condition, threshold in ARMS:
            print(f'[{time.strftime("%H:%M:%S")}] hour_slot={hour_slot}, key={hour_key}, condition={condition}, threshold={threshold}')
            rows.append(fit_one_hour_arm(hour_slot, hour_key, condition, threshold))
            partial = pd.concat(rows, ignore_index=True)
            full_mod.round_numeric(partial, 6).to_csv(RESULTS_CSV, index=False, float_format='%.6f')
            print('  saved partial:', RESULTS_CSV, 'rows=', len(partial))
    return pd.concat(rows, ignore_index=True)

In [ ]:
if RUN_FITS:
    if RESULTS_CSV.exists() and not FORCE_REFIT:
        print('Loading cached results:', RESULTS_CSV)
        results_df = pd.read_csv(RESULTS_CSV)
    else:
        results_df = run_experiment()
        full_mod.round_numeric(results_df, 6).to_csv(RESULTS_CSV, index=False, float_format='%.6f')
        print('Saved:', RESULTS_CSV)
else:
    if RESULTS_CSV.exists():
        results_df = pd.read_csv(RESULTS_CSV)
        print('Loaded existing results:', RESULTS_CSV)
    else:
        results_df = pd.DataFrame()
        print('Set RUN_FITS=True to run the July 3 eight-hour experiment.')

results_df.shape

## First Checks

The two most important checks are:
- whether QC actually computed whitening residuals: `qc_max_abs_whitened` should not be all missing in the QC arm;
- whether many first fits are still `success=False`; if so, check `qc_initial_success` versus `qc_refit` rather than relying only on SciPy's success flag.

In [ ]:
if results_df.empty:
    raise RuntimeError('No results yet. Run the fit cell first.')

results_df['success_bool'] = results_df['success'].astype(str).str.lower().isin(['true', '1'])
for col in ['qc_refit', 'qc_initial_success']:
    if col in results_df.columns:
        results_df[col + '_bool'] = results_df[col].astype(str).str.lower().isin(['true', '1'])

cols = [
    'condition', 'hour_slot_selected', 'tile_id', 'success', 'qc_initial_success', 'qc_refit',
    'n_fit', 'n_qc_removed', 'qc_max_abs_whitened', 'loss', 'nll', 'nugget', 'smooth',
    'range_lat', 'range_lon', 'message'
]
show_cols = [c for c in cols if c in results_df.columns]
results_df[show_cols].head(16)

In [ ]:
summary_cols = {
    'n_tiles': ('tile_id', 'count'),
    'success_rate': ('success_bool', 'mean'),
    'loss_mean': ('loss', 'mean'),
    'loss_median': ('loss', 'median'),
    'nll_mean': ('nll', 'mean'),
    'nugget_mean': ('nugget', 'mean'),
    'nu_mean': ('smooth', 'mean'),
    'range_lat_mean': ('range_lat', 'mean'),
    'range_lon_mean': ('range_lon', 'mean'),
}
if 'n_qc_removed' in results_df.columns:
    summary_cols['n_qc_removed_sum'] = ('n_qc_removed', 'sum')
    summary_cols['n_qc_removed_max'] = ('n_qc_removed', 'max')
if 'qc_max_abs_whitened' in results_df.columns:
    summary_cols['qc_max_abs_whitened_max'] = ('qc_max_abs_whitened', 'max')
if 'qc_refit_bool' in results_df.columns:
    summary_cols['qc_refit_rate'] = ('qc_refit_bool', 'mean')
if 'qc_initial_success_bool' in results_df.columns:
    summary_cols['qc_initial_success_rate'] = ('qc_initial_success_bool', 'mean')

arm_summary = results_df.groupby('condition').agg(**summary_cols).reset_index()
arm_summary

## Paired Deltas

Everything below compares the same hour/tile under `qc_w10` minus `no_qc`. Loss improvement should not be over-interpreted by itself because the QC arm fits after removing extreme points; parameter stability and QC counts matter more.

In [ ]:
PAIR_INDEX = ['hour_slot_selected', 'hour_key', 'tile_id', 'tile_y', 'tile_x']
METRICS = ['loss', 'nll', 'nugget', 'smooth', 'range_lat', 'range_lon', 'sigmasq', 'n_fit']
METRICS = [m for m in METRICS if m in results_df.columns]

no = results_df[results_df['condition'] == 'no_qc'].set_index(PAIR_INDEX)
qc = results_df[results_df['condition'] == 'qc_w10'].set_index(PAIR_INDEX)
paired = no[[c for c in METRICS if c in no.columns]].join(
    qc[[c for c in METRICS if c in qc.columns]],
    how='inner',
    lsuffix='_no_qc',
    rsuffix='_qc',
)

for metric in METRICS:
    paired[f'delta_{metric}'] = paired[f'{metric}_qc'] - paired[f'{metric}_no_qc']

qc_extra = [c for c in ['n_qc_removed', 'qc_max_abs_whitened', 'qc_refit', 'qc_initial_success', 'message'] if c in qc.columns]
paired = paired.join(qc[qc_extra].add_suffix('_qc'), how='left')
paired = paired.reset_index()
full_mod.round_numeric(paired, 6).to_csv(PAIRED_CSV, index=False, float_format='%.6f')
print('Saved paired deltas:', PAIRED_CSV)
paired.head(16)

In [ ]:
delta_cols = [c for c in paired.columns if c.startswith('delta_')]
paired[delta_cols + [c for c in ['n_qc_removed_qc', 'qc_max_abs_whitened_qc'] if c in paired.columns]].describe().T

## Plots

In [ ]:
def savefig(name):
    path = FIG_DIR / name
    plt.savefig(path, dpi=180, bbox_inches='tight')
    print('saved:', path)
    return path


def heatmap_hour_tile(df, value_col, title, cmap='viridis', center_zero=False):
    grid = df.pivot_table(index='hour_slot_selected', columns='tile_id', values=value_col, aggfunc='mean')
    grid = grid.reindex(index=range(N_HOURS), columns=range(TILE_Y * TILE_X))
    fig, ax = plt.subplots(figsize=(9, 4.8))
    arr = grid.to_numpy(dtype=float)
    if center_zero and np.isfinite(arr).any():
        vmax = np.nanmax(np.abs(arr))
        vmin = -vmax
    else:
        vmin = vmax = None
    im = ax.imshow(arr, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel('tile_id')
    ax.set_ylabel('July 3 hour slot')
    ax.set_xticks(np.arange(TILE_Y * TILE_X))
    ax.set_yticks(np.arange(N_HOURS))
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            if np.isfinite(arr[i, j]):
                ax.text(j, i, f'{arr[i, j]:.3g}', ha='center', va='center', color='white', fontsize=8)
    fig.colorbar(im, ax=ax, shrink=0.85)
    return fig, ax

In [ ]:
if 'n_qc_removed_qc' in paired.columns:
    heatmap_hour_tile(paired, 'n_qc_removed_qc', 'QC arm: number removed by |w| > 10', cmap='magma')
    savefig('heatmap_n_qc_removed.png')
    plt.show()

if 'qc_max_abs_whitened_qc' in paired.columns:
    heatmap_hour_tile(paired, 'qc_max_abs_whitened_qc', 'QC arm: max |whitened residual| before removal', cmap='magma')
    savefig('heatmap_qc_max_abs_whitened.png')
    plt.show()

In [ ]:
for metric in ['loss', 'nugget', 'smooth', 'range_lat', 'range_lon']:
    col = f'delta_{metric}'
    if col not in paired.columns:
        continue
    heatmap_hour_tile(paired, col, f'QC - no QC: {metric}', cmap='coolwarm', center_zero=True)
    savefig(f'heatmap_delta_{metric}.png')
    plt.show()

In [ ]:
plot_metrics = [m for m in ['loss', 'nugget', 'smooth', 'range_lat', 'range_lon'] if f'{m}_no_qc' in paired.columns and f'{m}_qc' in paired.columns]
fig, axes = plt.subplots(1, len(plot_metrics), figsize=(4.0 * len(plot_metrics), 3.7), constrained_layout=True)
if len(plot_metrics) == 1:
    axes = [axes]
for ax, metric in zip(axes, plot_metrics):
    x = paired[f'{metric}_no_qc']
    y = paired[f'{metric}_qc']
    ax.scatter(x, y, s=28, alpha=0.75)
    finite = np.isfinite(x) & np.isfinite(y)
    if finite.any():
        lo = min(float(x[finite].min()), float(y[finite].min()))
        hi = max(float(x[finite].max()), float(y[finite].max()))
        ax.plot([lo, hi], [lo, hi], color='black', linewidth=1)
    ax.set_title(metric)
    ax.set_xlabel('no QC')
    ax.set_ylabel('QC |w|>10')
savefig('paired_scatter_noqc_vs_qc.png')
plt.show()

In [ ]:
if 'delta_loss' in paired.columns:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(paired['delta_loss'].dropna(), bins=24, color='steelblue', edgecolor='white')
    ax.axvline(0, color='black', linewidth=1)
    ax.set_title('QC - no QC loss delta by hour/tile')
    ax.set_xlabel('delta loss')
    ax.set_ylabel('count')
    savefig('hist_delta_loss.png')
    plt.show()

## Interpretation Checklist

Use this as the first-pass readout:

- If `n_qc_removed` is almost always zero, the `|w| > 10` rule is harmless/conservative for this day.
- If a few tiles have removals and `nugget` drops there, the original nugget was likely absorbing spike-like artifacts.
- If `nu` (`smooth`) changes a lot only in tiles with removals, outliers were affecting smoothness estimation.
- If many first fits have `qc_initial_success=False`, the optimizer budget is still too small; compare finite losses cautiously and consider increasing `MAXITER`.
- Training loss alone is not a clean performance metric because the QC arm has fewer effective observations after removing extreme points.

In [ ]:
report = {
    'results_csv': str(RESULTS_CSV),
    'paired_csv': str(PAIRED_CSV),
    'fig_dir': str(FIG_DIR),
    'n_rows': int(len(results_df)),
    'n_pairs': int(len(paired)),
}
report